# Flow Matching Policy：条件二维轨迹生成

**Transformer-to-VLA · Chapter 04 / Generative Policies · Flow Matching Policy**

这一节与 `01_DDPM_Policy.ipynb` 使用**相同的训练数据与条件**：随机起点、终点及长度 `H=16` 的二维轨迹。我们只替换生成方法，以便直接比较 **DDPM 噪声预测**与 **Flow Matching 速度预测**。

学习目标：
1. 理解条件流匹配的插值路径、目标速度和 MSE 目标。
2. 使用 PyTorch 从零训练一个简单的 *conditioned velocity field*。
3. 从高斯噪声经 ODE 积分得到完整轨迹，并可视化生成顺序和中间状态。
4. 理解生成时间 $t\in[0,1]$ 与轨迹长度 $H$ 的区别，以及它与机器人策略的联系。

> 本教程只做低维轨迹生成，不涉及图像生成，也不声称训练出的模型已经具有避障或动力学安全保证。

## 1. 导入依赖、固定随机种子

只使用常见的 `torch`、`numpy`、`matplotlib`；可以在本地 VS Code / Jupyter 中从上到下运行。

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from matplotlib.collections import LineCollection

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. 训练数据：带条件的二维轨迹

每条训练样本都有：

- 条件 $o=[x_s,y_s,x_g,y_g]\in\mathbb R^4$：起点与终点。
- 干净动作片段 $A_1\in\mathbb R^{H\times 2}$：16 个二维 waypoint。

为保持与 Diffusion Policy Notebook 相同的设置，先生成起终点之间的线性插值，再添加轻微扰动。这个玩具数据的轨迹基本是直线，**没有障碍物，也没有多种绕行模式**；因此仅用于理解生成机制。

In [ ]:
H = 16

def generate_data(batch_size):
    start = torch.rand(batch_size, 2, device=device) * 2 - 1
    goal = torch.rand(batch_size, 2, device=device) * 2 - 1
    obs = torch.cat([start, goal], dim=1)                 # [B, 4]
    ratio = torch.linspace(0, 1, H, device=device)       # [H]
    trajectory = (start[:, None, :] * (1 - ratio[None, :, None])
                  + goal[:, None, :] * ratio[None, :, None])
    trajectory = trajectory + 0.02 * torch.randn_like(trajectory)
    return obs, trajectory                                # [B,4], [B,H,2]

obs_preview, actions_preview = generate_data(4)
print("Observation shape:", tuple(obs_preview.shape))
print("Trajectory shape:", tuple(actions_preview.shape))

fig, ax = plt.subplots(figsize=(5, 5))
for obs_i, traj_i in zip(obs_preview.cpu().numpy(), actions_preview.cpu().numpy()):
    ax.plot(traj_i[:, 0], traj_i[:, 1], '.-', alpha=0.75)
    ax.scatter(*obs_i[:2], marker='o', s=55)
    ax.scatter(*obs_i[2:], marker='x', s=55)
ax.set(title='Toy training trajectories (H=16)', xlabel='x', ylabel='y')
ax.set_aspect('equal', adjustable='box'); ax.grid(alpha=.2)
plt.show()

## 3. Conditional Flow Matching：从随机噪声到轨迹

给定一条训练轨迹 $A_1$，独立采样 $A_0\sim\mathcal N(0,I)$，并采样生成时间 $t\sim U(0,1)$。定义**线性插值路径**：

$$
A_t=(1-t)A_0+tA_1
$$

由于 $A_0,A_1$ 固定时轨迹关于 $t$ 是线性的，目标速度是：

$$
u^*=\frac{d A_t}{dt}=A_1-A_0$$

我们训练带条件的网络 $v_\theta(A_t,t,o)$ 预测这一速度：

$$
\mathcal L_{\mathrm{FM}}(\theta)=
\mathbb E_{o,A_1,A_0,t}\left[\left\|v_\theta(A_t,t,o)-(A_1-A_0)\right\|_2^2\right].
$$

**注意**：网络训练时输入是随机中间状态 $A_t$，并不会拿到 $A_1$；$A_1$ 只用于构造训练目标。这里是 *independent coupling* 的简单 Conditional Flow Matching 教学形式，并非唯一的概率路径。

In [ ]:
def flow_matching_batch(obs, clean_actions):
    B = clean_actions.shape[0]
    a0 = torch.randn_like(clean_actions)          # 起点分布 p0 = N(0,I)
    t = torch.rand(B, device=device)              # 连续生成时间 [B]
    t_view = t[:, None, None]                     # 可广播到 [B,H,2]
    a_t = (1 - t_view) * a0 + t_view * clean_actions
    target_velocity = clean_actions - a0
    return a_t, t, target_velocity

a_t_demo, t_demo, velocity_demo = flow_matching_batch(obs_preview, actions_preview)
print('A_t:', a_t_demo.shape, 't:', t_demo.shape,
      'target_velocity:', velocity_demo.shape)

## 4. 速度场网络：MLP + Sinusoidal Time Embedding

网络输入为展平的 $A_t$（$H\times2$）、连续时间 $t$ 的嵌入、条件 $o$；输出同形状的速度场 $v_\theta\in\mathbb R^{H\times2}$。

为让实验容易理解，这里先不用 Transformer 或 U-Net。正弦时间嵌入与 DDPM 版本思想相同，但 FM 的 $t$ 是 **连续区间 $[0,1]$** 中的数，而不是整数扩散步。

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, t):
        # t: [B], continuous from 0 to 1
        half = self.dim // 2
        scales = torch.exp(
            torch.arange(half, device=t.device, dtype=t.dtype)
            * (-math.log(10000) / max(half - 1, 1))
        )
        angles = t[:, None] * scales[None, :] * (2 * math.pi)
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)

class VelocityPredictor(nn.Module):
    def __init__(self, horizon=16, hidden_dim=128, time_dim=32):
        super().__init__()
        self.horizon = horizon
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.net = nn.Sequential(
            nn.Linear(horizon * 2 + time_dim + 4, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, horizon * 2),
        )

    def forward(self, a_t, t, obs):
        B = a_t.shape[0]
        x = torch.cat([a_t.reshape(B, -1), self.time_embed(t), obs], dim=-1)
        return self.net(x).reshape(B, self.horizon, 2)

model = VelocityPredictor(horizon=H).to(device)
print('Parameter count:', sum(p.numel() for p in model.parameters()))
print('One forward pass:', model(a_t_demo, t_demo, obs_preview).shape)

## 5. 训练速度场

每一次迭代：生成训练轨迹 $A_1$ → 采样 $A_0$ 与 $t$ → 构造 $A_t$ 和目标速度 → 使用 **MSE** 更新网络。

> 默认训练 `TRAIN_STEPS=5000`；CPU 也能运行，但可能较慢。若只想快速测试完整流程，可以临时设置为 `200`，但生成质量会下降。

In [ ]:
TRAIN_STEPS = 5000
BATCH_SIZE = 256
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_history = []
model.train()
for step in range(TRAIN_STEPS):
    obs, clean_actions = generate_data(BATCH_SIZE)
    a_t, t, target_velocity = flow_matching_batch(obs, clean_actions)
    predicted_velocity = model(a_t, t, obs)
    loss = torch.mean((predicted_velocity - target_velocity) ** 2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

    if (step + 1) % 500 == 0 or step == 0:
        print(f'Step {step+1:5d}/{TRAIN_STEPS}, Loss: {loss.item():.5f}')

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(loss_history, lw=1)
ax.set(xlabel='Training step', ylabel='Velocity MSE', title='Flow Matching training loss')
ax.grid(alpha=.25)
plt.show()

## 6. 使用 Euler ODE 积分生成 Action Chunk

训练好速度场后，给定条件 $o$，从全新的随机噪声 $A(0)\sim\mathcal N(0,I)$ 开始，数值求解常微分方程：

$$
\frac{dA(t)}{dt}=v_\theta(A(t),t,o),\qquad t:0\to1.
$$

Euler 离散更新：

$$
A_{k+1}=A_k+\Delta t\,v_\theta(A_k,t_k,o),\qquad \Delta t=1/N.
$$

这里 `NUM_ODE_STEPS` 是**生成时 ODE 积分的步数**；`H=16` 是**每条机器人动作片段的 waypoint 数**，二者完全不同。

In [ ]:
@torch.no_grad()
def sample_flow_policy(obs, num_ode_steps=50, return_snapshots=False, initial_noise=None):
    model.eval()
    B = obs.shape[0]
    actions = (torch.randn(B, H, 2, device=obs.device)
               if initial_noise is None else initial_noise.clone())
    snapshots = {0: actions.detach().cpu().clone()} if return_snapshots else None
    dt = 1.0 / num_ode_steps
    for k in range(num_ode_steps):
        t = torch.full((B,), k * dt, device=obs.device)
        actions = actions + dt * model(actions, t, obs)
        if return_snapshots and (k + 1) in {num_ode_steps // 4, num_ode_steps // 2,
                                            3 * num_ode_steps // 4, num_ode_steps}:
            snapshots[k+1] = actions.detach().cpu().clone()
    return (actions, snapshots) if return_snapshots else actions

NUM_ODE_STEPS = 50
obs_test = torch.tensor([[-0.85, -0.65, 0.80, 0.75]], device=device)
generated, snapshots = sample_flow_policy(obs_test, NUM_ODE_STEPS, True)
print('Generated action chunk:', tuple(generated.shape))

## 7. 可视化生成轨迹：显示起点、终点和动作顺序

轨迹点按时间顺序渐变着色，绿色表示条件起点，橙色表示条件终点。注意首尾 waypoint 不是被硬编码到起终点上的；它们由条件网络学习，因此仍可能存在误差。

In [ ]:
def draw_trajectory(ax, points, start, goal, title=None):
    points = np.asarray(points)
    segments = np.stack([points[:-1], points[1:]], axis=1)
    line = LineCollection(segments, cmap='viridis', linewidth=2.7)
    line.set_array(np.linspace(0, 1, len(segments)))
    ax.add_collection(line)
    ax.scatter(points[:, 0], points[:, 1], c=np.linspace(0, 1, len(points)),
               cmap='viridis', s=24, zorder=3)
    ax.scatter(*start, c='tab:green', s=90, marker='o', label='Start', zorder=5)
    ax.scatter(*goal, c='tab:orange', s=110, marker='*', label='Goal', zorder=5)
    ax.autoscale(); ax.set_aspect('equal', adjustable='box')
    ax.set(xlabel='x', ylabel='y', title=title)
    ax.grid(alpha=.2); ax.legend()

points = generated[0].detach().cpu().numpy()
condition = obs_test[0].cpu().numpy()
fig, ax = plt.subplots(figsize=(6, 5))
draw_trajectory(ax, points, condition[:2], condition[2:], 'Flow Matching generated trajectory')
plt.show()

## 8. 观察 ODE 生成过程

下面展示**生成时间**从 $t=0$ 到 $t=1$ 的中间轨迹。早期轨迹近似纯随机噪声，之后逐步符合起终点条件。这里横轴和纵轴仍是机器人二维平面坐标。

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
keys = [0, NUM_ODE_STEPS // 4, NUM_ODE_STEPS // 2,
        3 * NUM_ODE_STEPS // 4, NUM_ODE_STEPS]
for ax, k in zip(axes, keys):
    action_points = snapshots[k][0].numpy()
    ax.plot(action_points[:, 0], action_points[:, 1], 'o-', markersize=3, lw=1.4)
    ax.scatter(*condition[:2], c='tab:green', marker='o', s=45)
    ax.scatter(*condition[2:], c='tab:orange', marker='*', s=60)
    ax.set_title(f't={k / NUM_ODE_STEPS:.2f}')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=.2)
plt.suptitle('Trajectory evolution during Flow Matching ODE sampling')
plt.tight_layout(); plt.show()

## 9. 简单质量检查：端点误差与平滑性

验证在一批**新随机起终点**上生成的轨迹。这里只统计教学指标：首尾 waypoint 到条件起终点的误差和轨迹二阶差分，不把这些指标当作真实机器人任务成功率。

In [ ]:
obs_eval, _ = generate_data(128)
with torch.no_grad():
    gen_eval = sample_flow_policy(obs_eval, num_ode_steps=NUM_ODE_STEPS)
    start_error = torch.linalg.vector_norm(gen_eval[:, 0] - obs_eval[:, :2], dim=-1).mean()
    goal_error = torch.linalg.vector_norm(gen_eval[:, -1] - obs_eval[:, 2:], dim=-1).mean()
    second_difference = gen_eval[:, 2:] - 2 * gen_eval[:, 1:-1] + gen_eval[:, :-2]
    smoothness = second_difference.square().mean()
print(f'Start error: {start_error.item():.4f}')
print(f'Goal error: {goal_error.item():.4f}')
print(f'Smoothness (second-difference MSE): {smoothness.item():.6f}')

## 10. 与 DDPM Diffusion Policy 对照

| 项目 | DDPM Policy | Flow Matching Policy（本实验） |
|---|---|---|
| 训练样本 | 起终点 + 16 个二维 waypoint | **完全相同** |
| 网络输入 | $A_t,t,o$ | $A_t,t,o$ |
| 监督目标 | 注入的噪声 $\epsilon$ | 构造路径的速度 $A_1-A_0$ |
| 损失 | Noise MSE | Velocity MSE |
| 生成初始化 | 高斯噪声 | 高斯噪声 |
| 采样 | DDPM 逐步反向采样 | Euler ODE 积分 |
| 输出 | $[B,16,2]$ | $[B,16,2]$ |

**易混淆的问题**：
1. 为什么目标速度 $A_1-A_0$ 虽然不显式依赖 $t$，模型仍要输入 $t$？因为同一个 $A_t$ 在不同时间可能需要不同的**条件平均速度场**。
2. Flow Matching 是否每次只生成下一个 waypoint？**不是**。这里一次生成整个 $[H,2]$ 片段。
3. Flow Matching 天然保证碰撞安全和物理可执行吗？**不保证**。这里生成的是 waypoint 教学示例，并非底层速度或力矩指令。
4. 真实 VLA 使用什么作为条件？视觉 Token、语言指令、机器人本体状态等；这里仅使用起终点。

## 11. 可以自己做的实验

- 将 `NUM_ODE_STEPS` 改为 `5 / 10 / 20 / 50 / 100`，比较生成质量与推理开销。
- 固定条件，采样 8 个不同噪声初值，看看是否产生不同轨迹。
- 增大或减小 `H`，理解轨迹序列维度与模型输入输出的关系。
- 与 DDPM Notebook 使用**相同随机数据分布**和相近网络规模比较；这不是严谨的速度或质量基准，需控制训练算力、采样步数等变量。

**下一阶段**：为动作生成加入视觉 / 语言等条件，以及更真实的机器人动作定义，这些留在后续 VLA 章节学习。